In [1]:
import numpy as np
import pandas as pd

# Create Data Ingestion Function

In [3]:
import seaborn as sns
df=sns.load_dataset("titanic")
df.head()



,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


# Create a Data Validation Function

In [4]:
def data_validation(df):
   
    print("="*50)
    print("data validation report")
    print("="*50)
   
    #shape
    print(f"data shape:{df.shape}")
    
    print(f"data null values:\n{df.isnull().sum()}")

    print(f"duplicates values:{df.duplicated().sum()}")
    
    dtypes=df.dtypes
    print(f"data types:\n{dtypes}")
     
    if "target" in df.columns:
        print("target columns is present in the dataset")
    else:
        print("target columns is not present in the dataset")

    return df


In [7]:
df=data_validation(df)

data validation report
data shape:(891, 15)
data null values:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
duplicates values:107
data types:
survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object
target columns is not present in the dataset


In [11]:
from sklearn.model_selection import train_test_split

def split_data(df):


    y = df[["survived"]]
    X = df.drop("survived", axis=1)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    print("Train Test Split Completed")

    return X_train, X_test, y_train, y_test


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

def data_transformation(X_train, X_test):

    numerical_columns = X_train.select_dtypes(
        include=["int64", "float64"]
    ).columns

    categorical_columns = X_train.select_dtypes(
        include=["object","category","bool"]
    ).columns

    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

    X_train = preprocessor.fit_transform(X_train)

    X_test = preprocessor.transform(X_test)

    print("Data Transformation Completed")

    return X_train, X_test, preprocessor

In [13]:

df = data_validation(df)
X_train, X_test, y_train, y_test = split_data(df)

X_train, X_test, preprocessor = data_transformation(
    X_train,
    X_test
)


data validation report
data shape:(891, 15)
data null values:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
duplicates values:107
data types:
survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object
target columns is not present in the dataset
Train Test Split Completed
Data Transformation Completed


# Create Model Training Function

In [19]:
from sklearn.tree import DecisionTreeRegressor

def model_training(X_train, y_train):
    """
    Train the Decision Tree model.
    """

    # Create model
    model = DecisionTreeRegressor(
        max_depth=3,
        random_state=42
    )

    # Train model
    model.fit(X_train, y_train)

    print("="*50)
    print("MODEL TRAINING COMPLETED")
    print("="*50)

    return model

In [15]:
model = model_training(X_train, y_train)

MODEL TRAINING COMPLETED


In [16]:
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

def model_evaluation(model, X_test, y_test):
    """
    Evaluate the trained model.
    """
    # Prediction
    y_pred = model.predict(X_test)
    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("=" * 50)
    print("MODEL EVALUATION")
    print("=" * 50)
    return mae,r2

In [18]:
mae,r2=(model_evaluation(model,X_test,y_test))
print(mae,r2)

MODEL EVALUATION
0.0 1.0
